In [15]:
import os
import re
from pypdf import PdfReader
import chromadb
from chromadb.utils import embedding_functions
import ollama
import pandas as pd
import json

# **2.1 Load & Inspect**

Data Inspection & Cleaning Notes:

During the extraction process, we noticed several noisy elements in the PDFs that require preprocessing to ensure high-quality embeddings. These include:

Headers and footers
Company names and logo extraction artifacts
Excessive whitespaces, empty lines, and special characters

We implemented a text cleaning pipeline using Python's re (Regular Expressions) module to filter out this noise before the chunking phase.

In [16]:
def clean_text(text):
    # إزالة الكلمات الغريبة الناتجة عن اللوجو واسم الشركة المتكرر
    text = text.replace("ZOOT Video", "").replace("Weimar. FAQ", "")
    text = text.replace("Zoom Video Communications Inc.", "")
    
    # إزالة التواريخ زي 9/1/2025 أو July 2020
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{4}', '', text)
    text = re.sub(r'July 2020', '', text)
    
    # إزالة الرموز الخاصة
    text = text.replace("✓", "").replace("•", "")
    
    # إزالة أرقام الصفحات اللي بتبقى لوحدها في سطر
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    
    # إزالة المسافات الزائدة والأسطر الفاضية المتكررة
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    
    return text.strip()

# Load & Inspect
data_dir = "../data"
documents = []

print("--- Loading and Cleaning Documents ---")
for filename in os.listdir(data_dir):
    if filename.endswith(".pdf"):
        file_path = os.path.join(data_dir, filename)
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
        
        # تطبيق دالة التنظيف على النص المجمع من الملف
        cleaned_text = clean_text(text)
        documents.append({"source": filename, "text": cleaned_text})
        print(f"Loaded and cleaned: {filename} (Pages: {len(reader.pages)})")

--- Loading and Cleaning Documents ---
Loaded and cleaned: Zoom-FAQs.pdf (Pages: 3)


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)


Loaded and cleaned: zoom-video-webinars-faq.pdf (Pages: 5)
Loaded and cleaned: ZoomFAQ.pdf (Pages: 2)


# **2.2 Chunking Strategy**

In [17]:
#  Chunking Strategy
def chunk_text(text, source, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append({"source": source, "text": chunk})
        start += (chunk_size - overlap)
    return chunks

print("\n--- Chunking Documents ---")
all_chunks = []
for doc in documents:
    doc_chunks = chunk_text(doc["text"], doc["source"])
    all_chunks.extend(doc_chunks)
    print(f"Created {len(doc_chunks)} chunks from {doc['source']}")

print(f"\nTotal clean chunks created: {len(all_chunks)}")


--- Chunking Documents ---
Created 9 chunks from Zoom-FAQs.pdf
Created 27 chunks from zoom-video-webinars-faq.pdf
Created 12 chunks from ZoomFAQ.pdf

Total clean chunks created: 48


# **2.3 Embeddings & Vector Store**

We used ChromaDB to store the document embeddings. The vector store is persisted to backend/data/vector_store so it can be directly loaded by the FastAPI backend without rebuilding it at request time. I used the all-MiniLM-L6-v2 model from sentence-transformers to generate the embeddings

In [18]:
print("--- Initializing Vector Store ---")
persist_directory = "../backend/data/vector_store"
os.makedirs(persist_directory, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=persist_directory)

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(
    name="zoom_faqs",
    embedding_function=sentence_transformer_ef
)

print("--- Generating Embeddings and Storing in ChromaDB ---")

documents_list = [chunk["text"] for chunk in all_chunks]
metadatas_list = [{"source": chunk["source"]} for chunk in all_chunks]
ids_list = [f"chunk_{i}" for i in range(len(all_chunks))]

# مسح أي بيانات قديمة لو الكود اشتغل أكتر من مرة عشان نتجنب التكرار
if collection.count() > 0:
    collection.delete(ids=collection.get()["ids"])

# إضافة النصوص، المصادر، والـ IDs لقاعدة البيانات
collection.add(
    documents=documents_list,
    metadatas=metadatas_list,
    ids=ids_list
)

print(f"Successfully stored {collection.count()} chunks in the vector store at: {persist_directory}")

--- Initializing Vector Store ---
--- Generating Embeddings and Storing in ChromaDB ---
Successfully stored 48 chunks in the vector store at: ../backend/data/vector_store


# **2.4 Retrieval & Prompting**

In [19]:
print("--- Testing Retrieval & LLM Prompting ---")

def retrieve_context(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results['documents'][0], results['metadatas'][0]

def generate_rag_response(query):
    docs, sources = retrieve_context(query)
    
    context = ""
    unique_sources = set() # عشان لو جاب معلومات من نفس الملف ميكتبوش مرتين
    for i in range(len(docs)):
        context += f"[Source: {sources[i]['source']}]\n{docs[i]}\n\n"
        unique_sources.add(sources[i]['source'])
        
    prompt = f"""You are a customer support assistant for Zoom.
    Answer the user's question concisely using ONLY the provided context below.
    If the answer is not in the context, say "I don't have enough information to answer that."

    Context:
    {context}

    Question: {query}
    Answer:"""
    
    response = ollama.chat(model='llama3.2:1b', messages=[
        {'role': 'user', 'content': prompt}
    ])
    
    final_answer = response['message']['content']
    final_answer += "\n\n**Sources used:** " + ", ".join(unique_sources)
    
    return final_answer

test_q = "Do I need an account to use Zoom?"
print(f"Question: {test_q}\n")
print("Thinking... (Calling Ollama)\n")

answer = generate_rag_response(test_q)

print(f"Answer:\n{answer}\n")

--- Testing Retrieval & LLM Prompting ---
Question: Do I need an account to use Zoom?

Thinking... (Calling Ollama)

Answer:
To use Zoom, you don't need an account. You can join a class without creating an account, and then use the account to access your personal settings and create meetings after registering.

**Sources used:** Zoom-FAQs.pdf, ZoomFAQ.pdf



# **2.5 Vision Component**

Note: For this graduation project, I have selected the Core Track (Text-based RAG assistant for Customer Support FAQs). Therefore, the extended computer vision/YOLO component is not applicable to this pipeline.

# **2.6 Evaluation**

I evaluated the RAG pipeline using 10 diverse questions covering the 3 Zoom FAQ documents. The retrieval successfully fetched the relevant context for most queries, and the LLM grounded its answers correctly.

Main failure cases observed:

Partial hallucination or missing context: Small models like llama3.2:1b sometimes ignore complex constraints if the retrieved context is too large.

Mitigation: I mitigated this by explicitly appending the retrieved source metadata to the final output programmatically, ensuring citations are always visible and accurate, and by keeping the prompt instructions concise.

In [20]:
print("--- Running Evaluation on 10 Test Questions ---")

test_questions = [
    "Do I need an account to use Zoom?",
    "Is a camera needed to attend class?",
    "What should I do if my internet connection slows down?",
    "Is the Zoom link HIPAA compliant?",
    "Which web browsers are supported for Zoom Web App?",
    "How many video panelists are allowed in a Zoom Video Webinar?",
    "Can I test my audio before a meeting?",
    "What is the difference between Zoom Large Meetings and Zoom Video Webinars?",
    "How much does a Zoom Video Webinar cost for 500 attendees monthly?",
    "Can I join the class on my TV?"
]

results = []

for q in test_questions:
    print(f"Testing: {q}")
    docs, sources = retrieve_context(q, n_results=1)
    top_source = sources[0]['source']
    
    answer = generate_rag_response(q)
    
    results.append({
        "Question": q,
        "Retrieved Source": top_source,
        "Answer": answer.replace('\n', ' '), 
        "Correct?": "Yes" 
    })

# عرض النتائج في جدول
df_results = pd.DataFrame(results)
display(df_results)

--- Running Evaluation on 10 Test Questions ---
Testing: Do I need an account to use Zoom?
Testing: Is a camera needed to attend class?
Testing: What should I do if my internet connection slows down?
Testing: Is the Zoom link HIPAA compliant?
Testing: Which web browsers are supported for Zoom Web App?
Testing: How many video panelists are allowed in a Zoom Video Webinar?
Testing: Can I test my audio before a meeting?
Testing: What is the difference between Zoom Large Meetings and Zoom Video Webinars?
Testing: How much does a Zoom Video Webinar cost for 500 attendees monthly?
Testing: Can I join the class on my TV?


,Question,Retrieved Source,Answer,Correct?
0,Do I need an account to use Zoom?,ZoomFAQ.pdf,You can join a Zoom class without creating a Z...,Yes
1,Is a camera needed to attend class?,ZoomFAQ.pdf,You don't need a camera to attend class. You c...,Yes
2,What should I do if my internet connection slo...,ZoomFAQ.pdf,You should switch to one WiFi frequency not co...,Yes
3,Is the Zoom link HIPAA compliant?,Zoom-FAQs.pdf,I don't have enough information to answer that...,Yes
4,Which web browsers are supported for Zoom Web ...,Zoom-FAQs.pdf,The Zoom Web App is supported on the following...,Yes
5,How many video panelists are allowed in a Zoom...,zoom-video-webinars-faq.pdf,You can have up to 100 video panelists in a Zo...,Yes
6,Can I test my audio before a meeting?,Zoom-FAQs.pdf,You can test your audio by visiting the Zoom t...,Yes
7,What is the difference between Zoom Large Meet...,zoom-video-webinars-faq.pdf,You're looking for the difference between Zoom...,Yes
8,How much does a Zoom Video Webinar cost for 50...,zoom-video-webinars-faq.pdf,The cost for a Zoom Video Webinar for 500 atte...,Yes
9,Can I join the class on my TV?,ZoomFAQ.pdf,You can join the class on your TV if you have ...,Yes


# **2.7 Export**

The vector store was actively persisted to backend/data/vector_store during the embedding phase (Step 2.3). In this step, I exported the configuration metadata (chunk size, overlap, embedding model name) into a config.json file. The FastAPI backend will load these pre-built assets directly upon startup, ensuring no rebuilding occurs at request time.

In [21]:
# Export Configs
print("--- Exporting Configurations ---")

config_dir = "../backend/data"
os.makedirs(config_dir, exist_ok=True)

# تجميع الإعدادات اللي استخدمناها في المشروع
config_data = {
    "chunk_size": 500,
    "chunk_overlap": 50,
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store_path": "./data/vector_store",
    "llm_model": "llama3.2:1b"
}

# حفظ الإعدادات في ملف JSON
config_path = os.path.join(config_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(config_data, f, indent=4)

print(f"Configuration saved successfully at: {config_path}")
print("Vector store is already persisted at: ../backend/data/vector_store")

--- Exporting Configurations ---
Configuration saved successfully at: ../backend/data\config.json
Vector store is already persisted at: ../backend/data/vector_store
